# Agentic Phishing Email Detection and Explanation

Given a raw email, this notebook returns a **verdict** (phishing / suspicious / legitimate),
a **risk score** (0-100), **reasons** backed by evidence, and a **recommended action**.

Three agents work together:
- **Technical Analyst**: deterministic Python checks on headers and URLs (no LLM on a normal run).
- **Content Analyst**: an LLM that reads the text and identifies intent and social-engineering tactics.
- **Verdict and Explainer**: merges both reports. If they disagree, it asks one analyst a single follow-up question.

Run the cells top to bottom. Each stage ends with a **checkpoint** cell that tells you whether the stage worked.

## Stage 0: Config and setup

Everything you might want to change lives in the next cell. No other cell hardcodes model names or limits.

In [ ]:
# ---- Config: the only place to change models, limits and paths ----
PRIMARY_MODEL  = "gemini-3.1-flash-lite"   # used for ALL evaluation runs
FALLBACK_MODEL = "gemini-3.5-flash-lite"   # used only if the primary fails the smoke test

RPM_LIMIT       = 15     # requests per minute on the free tier
RPD_LIMIT       = 500    # requests per day (resets at midnight Pacific)
RPD_SAFETY_STOP = 450    # stop evaluation runs here to leave headroom

SEED             = 42
PROMPT_VERSION   = "v1"  # bump this when a prompt changes, so old cache entries are ignored
BODY_CHAR_LIMIT  = 6000  # email bodies are truncated to this before going to the LLM

# Score thresholds that turn the risk score into a label
PHISHING_THRESHOLD   = 70   # score >= 70 -> phishing
SUSPICIOUS_THRESHOLD = 40   # 40-69 -> suspicious, below 40 -> legitimate

DRIVE_ROOT = "/content/drive/MyDrive/phish_project"

CREWAI_VERSION = "1.15.28"

### Install

CrewAI is pinned to one version, with its `google-genai` extra so it can talk to Gemini directly.
It needs Python 3.10–3.13. If pip says it changed packages that were already loaded,
use **Runtime → Restart session**, then run the cells again from the top. (The install is quick the second time.)

In [ ]:
import sys
print("Python", sys.version.split()[0])
assert (3, 10) <= sys.version_info[:2] < (3, 14), "CrewAI 1.15.x needs Python 3.10-3.13"

!pip install -q "crewai[google-genai]=={CREWAI_VERSION}"

In [ ]:
import os

# Turn off CrewAI's anonymous telemetry; nothing about our runs needs to leave Colab except the Gemini calls.
os.environ["CREWAI_TELEMETRY_OPT_OUT"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"

import crewai, google.genai, pydantic
print("crewai      ", crewai.__version__)
print("google-genai", google.genai.__version__)
print("pydantic    ", pydantic.__version__)

# Google Drive holds the cache and results, so an interrupted run can resume.
from google.colab import drive
drive.mount("/content/drive")
for sub in ["data", "cache", "results"]:
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)
print("Drive folder:", DRIVE_ROOT)

# The API key comes from Colab Secrets (key icon in the left sidebar), never from a cell.
from google.colab import userdata
_key = userdata.get("GEMINI_API_KEY")
os.environ["GEMINI_API_KEY"] = _key
os.environ["GOOGLE_API_KEY"] = _key   # some libraries look for this name instead
del _key                              # don't leave it lying around in a variable
print("key loaded:", bool(os.environ.get("GEMINI_API_KEY")))

**✅ Checkpoint 0:** the cell above prints three version numbers, the Drive folder, and `key loaded: True`.
It must not print the key itself.

## Stage 1: Smoke test

This is the riskiest part of the project, so it runs before anything else is built. It tests three things:
1. The model names in the config exist on your account.
2. A plain call through CrewAI's Gemini integration works.
3. A CrewAI agent can return **structured output** (a pydantic object), on both the primary and the fallback model.

This stage uses about 3 requests of your daily quota.

In [ ]:
# 1. Do the configured model names exist? Listing models does not use generation quota.
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
available = sorted(m.name.removeprefix("models/") for m in client.models.list())

print("Flash Lite models on this key:")
for name in available:
    if "flash-lite" in name:
        print("  ", name)

for name in [PRIMARY_MODEL, FALLBACK_MODEL]:
    print(f"{name}: {'found' if name in available else 'NOT FOUND, fix the config cell'}")

In [ ]:
# 2. One plain call through CrewAI's native Gemini LLM.
from crewai import LLM

def make_llm(model_name):
    # temperature=0 makes answers as repeatable as the model allows
    return LLM(model=f"gemini/{model_name}", temperature=0)

reply = make_llm(PRIMARY_MODEL).call("Reply with the single word OK.")
print("Reply:", repr(reply))

### The Content Analyst's output schema

The Content Analyst must return this structure instead of free text. Writing it as a pydantic model
means Python checks every field and type for us. The same schema is used in the real system later.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

Intent = Literal["credential_theft", "payment_request", "personal_info", "malware_delivery",
                 "advance_fee", "account_verification", "benign", "other"]
TacticName = Literal["urgency", "authority", "fear", "reward", "scarcity", "curiosity"]

class Tactic(BaseModel):
    tactic: TacticName
    quote: str = Field(description="exact words copied from the email that show this tactic")
    explanation: str

class ContentReport(BaseModel):
    intent: Intent
    tactics: list[Tactic]
    content_risk: int = Field(ge=0, le=100, description="0 = clearly harmless, 100 = clearly malicious")
    summary: str

In [ ]:
# 3. A single agent and task that must return a ContentReport.
from crewai import Agent, Task, Crew

# Synthetic test email, written for this notebook. It is not a real person's email.
SMOKE_EMAIL = """Subject: Urgent: your account will be suspended

Dear customer,

We detected unusual sign-in activity on your account. To avoid permanent suspension,
verify your identity within 24 hours by logging in here: http://secure-paypa1-login.xyz/verify

Thank you,
Security Team"""

def smoke_test(model_name):
    analyst = Agent(
        role="Content Analyst",
        goal="Identify what an email is trying to make the reader do, and which manipulation tactics it uses.",
        backstory="You study phishing emails and describe them precisely, quoting the email as evidence.",
        llm=make_llm(model_name),
        allow_delegation=False,   # no handing work to other agents
        max_iter=1,               # one attempt, so one task = one request
        verbose=False,
    )
    task = Task(
        description=("Analyse this email. Quote its exact words as evidence for each tactic.\n\n"
                     "EMAIL:\n{email}"),
        expected_output="A ContentReport.",
        agent=analyst,
        output_pydantic=ContentReport,
    )
    result = Crew(agents=[analyst], tasks=[task], verbose=False).kickoff(inputs={"email": SMOKE_EMAIL})

    print(f"=== {model_name} ===")
    print("Token usage:", result.token_usage)
    if result.pydantic is None:
        print("Structured output FAILED. Raw reply:")
        print(result.raw)
        return None
    print(result.pydantic.model_dump_json(indent=2))
    return result.pydantic

smoke_results = {}
for name in [PRIMARY_MODEL, FALLBACK_MODEL]:
    try:
        smoke_results[name] = smoke_test(name)
    except Exception as e:   # show the error but keep going, so both models get tested
        print(f"=== {name} === ERROR: {type(e).__name__}: {e}")
        smoke_results[name] = None

In [ ]:
# ✅ Checkpoint 1
for name, report in smoke_results.items():
    print(f"{name}: {'PASS' if isinstance(report, ContentReport) else 'FAIL'}")

if isinstance(smoke_results.get(PRIMARY_MODEL), ContentReport):
    print("\nPrimary model works. Keep it for every evaluation run.")
else:
    print("\nPrimary model failed. Stop here and decide on a fallback before building further.")

**✅ Checkpoint 1:** the model names are found, the plain call replies with OK,
and the primary model shows `PASS` with a sensible report (intent `credential_theft`, tactics such as urgency and fear,
quotes copied from the email). Note the token usage. It feeds the budget estimate.